# Polydisperse Systems

From a target molecular-weight distribution to a packed, LAMMPS-ready box: sample the chains, assemble each one, type it, pack, export.

!!! note "Prerequisites"
    This guide requires `molcrafts-molpack` for the packing step. Familiarity with [Assembly](02_assembly.md) is assumed.

## From distribution to simulation box

Most polymer samples are polydisperse rather than monodisperse. This workflow starts from a target molecular-weight distribution and proceeds through explicit sampling, chain assembly, and packing to produce a simulation-ready system.

## Each monomer is a fragment with two ports

A styrene / methyl-acrylate copolymer grows by radical addition: each junction is a new C–C bond between the backbone carbons of neighbouring units, and one hydrogen leaves each side. That is exactly what a port expresses. Each unit is a fragment whose bonding descriptors sit on the two backbone carbons (`[<]` on one, `[>]` on the other), so `<` of one unit joins `>` of the next, head to tail.

`mp.builder.Fragment` names a ported template and gives each port a generation weight. A port's weight is its reactivity: when a chain end picks its next unit, units are chosen in proportion to the weight of the port they offer, so weights 8 on styrene and 2 on methyl acrylate give an 80:20 copolymer. Two methyl caps close the chain ends: `HEAD` (`C[>]`) caps an open `<`, `TAIL` (`[<]C`) an open `>`. A fragment knows the mass it adds to a chain — its heavy atoms plus the hydrogens they keep once every port is bonded — so nothing is computed by hand below.

In [ ]:
import molpy as mp

# One fragment per unit: its bonding descriptors are its ports, and the
# weight on each port is that port's reactivity.
BODIES = {
    "Sty": ("[<]CC(c1ccccc1)[>]", 8.0),  # -CH2-CH(Ph)-
    "MA": ("[<]CC(C(=O)OC)[>]", 2.0),  # -CH2-CH(COOCH3)-
    "HEAD": ("C[>]", 1.0),  # CH3- : caps an open "<"
    "TAIL": ("[<]C", 1.0),  # -CH3 : caps an open ">"
}
fragments = {}
for name, (body, weight) in BODIES.items():
    template = mp.io.smiles.read_smiles_fragment(body).to_template()
    fragments[name] = mp.builder.Fragment(name, template, [weight] * template.n_ports)

for name, fragment in fragments.items():
    print(
        f"{name:4s}: ports={fragment.n_ports}, adds {fragment.mass:.2f} g/mol "
        f"({fragment.heavy_atom_mass:.2f} g/mol heavy atoms)"
    )

## Sampling draws chain lengths from a statistical distribution

The polymer model has three layers. A `StochasticObject` holds the repeat units and the end groups and grows one chain: it starts from an end group, adds repeat units at the open end — chosen by port weight — until the chain reaches a size drawn from its `ChainSize`, then caps the last open port with the end group that fits it. A `ChainSize` is a `Distribution` together with the measure its draws count: `"repeat_units"`, `"mass"` (g/mol, hydrogens included) or `"heavy_atom_mass"`. A `Molecule` joins fixed fragments and stochastic objects end to end — here it is one object — and a `System` samples molecules until each kind reaches its `Amount` of the total.

Growth stops at the first unit that reaches the drawn target, so a chain overshoots a mass target by less than one unit, and every chain holds at least one repeat unit. The system stops at the first chain that reaches the system total: it overshoots by less than one chain, nothing is trimmed, and the amount is sampled as written. Four distributions are demonstrated below so that their shape differences become visible in the next section. Schulz–Zimm is a distribution of chain masses; the other three draw repeat-unit counts. A target counted in repeat units is a whole number — `Uniform(low=8, high=22)` draws each of 8 … 22 with equal probability, and a continuous draw is rounded.

In [ ]:
import numpy as np
from molpy.builder import Amount, ChainSize, Distribution, Molecule, StochasticObject, System

sizes = {
    "Schulz-Zimm": ChainSize(Distribution.SchulzZimm(mw=1500.0, mn=1400.0), "mass"),
    "Uniform": ChainSize(Distribution.Uniform(low=8.0, high=22.0), "repeat_units"),
    "Poisson": ChainSize(Distribution.Poisson(n=14.0), "repeat_units"),
    "Flory-Schulz": ChainSize(Distribution.FlorySchulz(a=0.08), "repeat_units"),
}


def copolymer(size):
    """One Sty/MA chain capped by HEAD and TAIL, grown to ``size``."""
    chain = StochasticObject(
        [fragments["Sty"], fragments["MA"]],
        [fragments["HEAD"], fragments["TAIL"]],
        size=size,
    )
    return Molecule([chain])


target_total_mass = 5e5  # g/mol, hydrogens included
results = {}
for name, size in sizes.items():
    system = System([(copolymer(size), Amount.Total(value=target_total_mass))], measure="mass")
    results[name] = system.sample(42).molecules

for name, chains in results.items():
    m = np.array([c.mass for c in chains])
    Mn = float(np.mean(m))
    Mw = float(np.sum(m**2) / np.sum(m))
    print(f"{name:15s}: {len(chains):4d} chains, Mn={Mn:.0f}, PDI={Mw / Mn:.3f}")

## Visualising the sampled ensembles reveals the distribution shape

The four panels below overlay sampled histograms against their theoretical curves. Schulz–Zimm is plotted as a probability density over chain mass; the other three as probability mass functions over the number of repeat units. Vertical dashed lines mark Mn and Mw for each ensemble.

In [ ]:
# docs: skip — needs matplotlib and scipy, which are not molpy runtime/doc dependencies
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats

# ── colour palette ──
CLR_HIST = "#6baed6"  # steel blue – sampled histogram
CLR_EDGE = "#3182bd"  # darker blue – histogram edge
CLR_THEO = "#e6550d"  # orange – theoretical curve
CLR_MN = "#31a354"  # green – Mn line
CLR_MW = "#de2d26"  # red – Mw line
CLR_BOX = "#f7f7f7"  # near-white – annotation box


def theory(distribution, n):
    """The distribution's density (mass) or probability mass (counts) at ``n``."""
    match distribution:
        case Distribution.SchulzZimm(mw=mw, mn=mn):
            z = mn / (mw - mn)
            return stats.gamma(a=z, scale=mn / z).pdf(n)
        case Distribution.Uniform(low=low, high=high):
            # each whole number of [low, high] is equally likely
            return ((n >= low) & (n <= high)) / (np.floor(high) - np.ceil(low) + 1)
        case Distribution.Poisson(n=mean):
            return stats.poisson(mean).pmf(n)
        case Distribution.FlorySchulz(a=a):
            return a * a * n * (1 - a) ** (n - 1)


def annotate_stats(ax, Mn, Mw, PDI, n_chains):
    txt = "\n".join(
        [
            rf"$M_n = {Mn:.0f}$ g/mol",
            rf"$M_w = {Mw:.0f}$ g/mol",
            rf"PDI $= {PDI:.3f}$",
            rf"$N = {n_chains}$",
        ]
    )
    ax.text(
        0.97, 0.97, txt, transform=ax.transAxes, ha="right", va="top",
        fontsize=6.5, linespacing=1.4, family="monospace",
        bbox=dict(boxstyle="round,pad=0.35", facecolor=CLR_BOX, edgecolor="0.75",
                  alpha=0.95, linewidth=0.6),
    )


fig, axes = plt.subplots(2, 2, figsize=(7.5, 6), constrained_layout=True)

for idx, (ax, (name, chains)) in enumerate(zip(axes.flatten(), results.items())):
    size = sizes[name]
    m = np.array([c.mass for c in chains])
    n = np.array([c.n_repeat_units for c in chains])
    Mn = float(np.mean(m))
    Mw = float(np.sum(m**2) / np.sum(m))

    if size.measure == "mass":
        iqr = np.subtract(*np.percentile(m, [75, 25]))
        bw = max(2.0 * iqr / len(m) ** (1 / 3), 20)
        bins = np.arange(m.min() - bw, m.max() + 2 * bw, bw)
        ax.hist(m, bins=bins, density=True, color=CLR_HIST, edgecolor=CLR_EDGE,
                linewidth=0.5, alpha=0.55)
        grid = np.linspace(max(0, m.min() * 0.3), m.max() * 1.3, 500)
        ax.plot(grid, theory(size.distribution, grid), color=CLR_THEO, linewidth=1.6,
                label="Theory")
        ax.axvline(Mn, color=CLR_MN, ls="--", lw=1, label=r"$M_n$")
        ax.axvline(Mw, color=CLR_MW, ls="--", lw=1, label=r"$M_w$")
        ax.set_xlabel(r"Chain mass $M$ (g mol$^{-1}$)", fontsize=8)
        ax.set_ylabel("Probability density", fontsize=8)
    else:
        lo, hi = int(n.min()), int(n.max())
        counts = np.bincount(n)[lo:]
        ax.bar(np.arange(lo, lo + len(counts)), counts / counts.sum(), width=1.0,
               align="center", color=CLR_HIST, edgecolor=CLR_EDGE, linewidth=0.5,
               alpha=0.55)
        support = np.arange(max(1, lo), max(hi, int(np.percentile(n, 99) * 1.15)) + 1)
        ax.plot(support, theory(size.distribution, support), "-o", color=CLR_THEO,
                markersize=2.2, linewidth=1.3, markeredgewidth=0, label="Theory")
        unit_mass = float(np.mean(m / np.maximum(n, 1)))
        ax.axvline(Mn / unit_mass, color=CLR_MN, ls="--", lw=1, label=r"$M_n$")
        ax.axvline(Mw / unit_mass, color=CLR_MW, ls="--", lw=1, label=r"$M_w$")
        ax.set_xlim(max(0, lo - 1), int(np.percentile(n, 99.5)) + 2)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel(r"Repeat units $n$", fontsize=8)
        ax.set_ylabel("Probability", fontsize=8)

    ax.set_title(name, fontsize=9.5, fontweight="bold", pad=6)
    ax.tick_params(labelsize=7)
    ax.spines[["top", "right"]].set_visible(False)
    annotate_stats(ax, Mn, Mw, Mw / Mn, len(chains))
    if idx == 0:
        ax.legend(fontsize=7, loc="upper left", framealpha=0.85)

plt.savefig("05_polydisperse_distributions.png", dpi=200, bbox_inches="tight")
plt.show()

## Each sampled chain is a site graph

`Molecule.sample(seed)` returns one chain as a site graph: a `CoarseGrain` with one site per placed fragment, whose `bead_type` is the fragment's name, and one bond per junction recording which port of each fragment it joins. `molecule.library()` maps those names to the fragments' templates — the library `mp.builder.Assembler` builds from, once `mp.conformer.Conformer` has embedded each template in 3D with hydrogens; the hydrogen on each port is its leaving atom.

The assembler places one copy of `library[bead_type]` per site and joins the two named ports of each bond, removing the two leaving hydrogens. `GrowthPlacer` needs no site positions: it grows the chain from its first unit, setting each copy's anchor where its parent's leaving hydrogen was. One assembler serves every chain, because both monomers and both caps come from one library. Every port is consumed, so the chains have no open ports left, and each built chain has the mass its sample reported.

In [ ]:
from molpy.core import Element


def mass(atoms):
    return sum(Element.by_number(a.get("atomic_number")).mass for a in atoms)


molecule = copolymer(sizes["Schulz-Zimm"])
conformer = mp.conformer.Conformer(seed=42)
library = {
    name: conformer.generate(template)[0]
    for name, template in molecule.library().items()
}
assembler = mp.builder.Assembler(library, mp.builder.GrowthPlacer())

# a few chains for this guide; a production run assembles every chain it sampled
atomistic_chains = [assembler.assemble(molecule.sample(seed)) for seed in range(5)]
for built in atomistic_chains:
    print(f"atoms={built.n_atoms:4d}  open ports={built.n_ports}  mass={mass(built.atoms):.1f} g/mol")

## The assembled chains are typed as whole molecules

Assembly assigns no force-field types. Each finished chain is an ordinary `mp.Atomistic`, so it is typed like any other molecule; the typifier's `forcefield()` then holds the parameters it assigned, ready for export.

In [ ]:
from molpy.ff.typifier import OplsAaTypifier

typifier = OplsAaTypifier(strict=True)
typed_chains = [typifier.typify(chain) for chain in atomistic_chains]
ff = typifier.forcefield()
print(f"typed {len(typed_chains)} chains, {sum(c.n_atoms for c in typed_chains)} atoms")

## Packing and exporting follow the same pattern as earlier guides

The box size follows from total molecular weight and target density. Each chain is added to the packer as an individual target with count 1; `mp.builder.PackingTemplate` hands the packer the chain's frame together with its `hydrogens` indices into that frame, which give the hydrogens a small packing radius. Which atoms are hydrogens is molpack's own default (element `H`), so the target is not told again. The packed frame is written as a LAMMPS data file together with the force field. The growth placer leaves bond lengths and close contacts to a later relaxation, which is why the LAMMPS script below minimises before any dynamics.

In [ ]:
# docs: skip — optional molcrafts-molpack; not a molpy runtime/doc dep
from pathlib import Path

from molpack import GencanPack, Target

total_mw = sum(mass(c.atoms) for c in typed_chains)
target_density = 0.05  # g/cm^3 (use ~1.0 for production)
# amu per cubic angstrom, so a mass in g/mol (numerically amu) gives angstrom^3
density = target_density * mp.core.UnitRegistry().factor("g/cm^3", "amu/angstrom^3")
volume = total_mw / density
box_length = volume ** (1 / 3)

box = mp.core.Cuboid([0.0, 0.0, 0.0], [box_length] * 3)
targets = []
for chain in typed_chains:
    template = mp.builder.PackingTemplate(chain)  # frame + hydrogen indices
    targets.append(
        Target(template.frame, count=1)
        .with_restraint(box)
        .with_atom_radius(template.hydrogens, 0.2)  # H relaxes away in early MD
    )
packed = GencanPack().with_seed(42).run(targets, max_loops=200).frame  # carries mol_id
packed.box = mp.Box(mp.core.Cell.lengths([box_length] * 3))

# the pair styles carry the 12 Å cutoff the OPLS-AA typifier declares

Path("05_output").mkdir(exist_ok=True)
mp.io.write_lammps_data(Path("05_output/system.data"), packed)
# the input script states `units` before `read_data`; LAMMPS refuses a second
# `units` line once the box exists, so the include leaves it out
mp.io.lammps.LammpsForcefieldWriter(packed, skip_units=True).write(
    Path("05_output/system.ff"), ff
)
print(f"packed: {packed['atoms'].n_rows} atoms, box: {box_length:.1f} A")

## The engine assembles a runnable input script from the exported data

Writing the data file is only half the story. To actually run the simulation, LAMMPS needs an input script that says how to read that file, which force field styles to activate, and what protocol to follow. MolPy models this through `LammpsEngine`, which pairs a `Script` object with subprocess management.

**A `Script` is an immutable value — a name, its text and an optional path —** that can be saved to disk without executing anything. This separation matters: you can inspect, modify, and version-control the script before committing to a run. When you are ready, `engine.run()` writes the script to the working directory and launches `lmp -in input.lmp -log log.lammps -screen none`.

The code below builds a minimal equilibration protocol for the packed system. It reads the `system.data` and `system.ff` written above. The include already declares every style, the mixing rule (geometric, as OPLS-AA defines it) and `special_bonds`, so the script does not repeat them: a `pair_style` of another kind issued after the include would discard the pair coefficients it just set. The script states `units` itself, which is why the include was written with `skip_units=True`. For long-range electrostatics, write the include with `mp.io.lammps.LammpsForcefieldWriter(frame, skip_pair_style=True)` too and declare `pair_style` and `kspace_style` in the script before `include`, as the [AmberTools guide](13_ambertools_integration.md) does; the include still sets the mixing rule and `special_bonds`.

In [ ]:
from molpy.engine import LammpsEngine, Script

# Build the LAMMPS input script from one block.
# Script.from_text() dedents and normalises the block.
lmp_script = Script.from_text(
    name="input",
    text="""
 # Polydisperse PS/PMA system — generated by MolPy
 units real
 atom_style full

 read_data system.data
 # every style, the mixing rule, special_bonds and the coefficients
 include system.ff

 # Energy minimisation before dynamics
 minimize 1.0e-4 1.0e-6 10000 100000

 timestep 1.0
 thermo 1000
 thermo_style custom step temp press etotal

 # NVT equilibration at 300 K
 fix nvt all nvt temp 300.0 300.0 100.0
 run 100000
 """,
)

# Save the script next to the data files without launching LAMMPS.
# check_executable=False lets the call succeed in notebooks where lmp
# may not be on PATH.
engine = LammpsEngine(check_executable=False)
script_path = lmp_script.save("05_output/input.lmp")
print("Input script written to:", script_path)
print(lmp_script.preview(max_lines=12))

# To run the simulation, replace the two lines above with:
# result = engine.run(lmp_script, workdir="05_output")
# print("Exit code:", result.returncode)

## The same system is one generative-BigSMILES string

The model above is the model of generative BigSMILES, and the notation writes all of it in one string: the repeat units and end groups of a stochastic object `{[] …; … []}`, each port's weight inside its bracket (`[<|8|]`), the size distribution after the object (`|schulz_zimm(1500, 1400)|`) and the system total after the molecule (`.|5e5|`). `mp.io.bigsmiles.read_bigsmiles` reads it and `to_system()` returns the same `System`, its fragments named by position (`"0.0.r0"` is repeat unit 0 of part 0 of molecule 0). The notation counts every size in **heavy-atom mass**, so these chains are lighter in hydrogens-included mass than the ones above.

In [ ]:
text = (
    "{[] [<|8|]CC(c1ccccc1)[>|8|], [<|2|]CC(C(=O)OC)[>|2|]; C[>], [<]C []}"
    "|schulz_zimm(1500, 1400)|.|5e5|"
)
system = mp.io.bigsmiles.read_bigsmiles(text).to_system()
chains = system.sample(42).molecules
heavy = np.array([c.heavy_atom_mass for c in chains])
Mn = float(np.mean(heavy))
Mw = float(np.sum(heavy**2) / np.sum(heavy))
print(f"{len(chains)} chains, heavy-atom Mn={Mn:.0f}, PDI={Mw / Mn:.3f}")
print(sorted(system.library()))

## Troubleshooting

| Step | Check |
|------|-------|
| `PolymerError` of kind `Stuck` | An open port matches no end group: give each port kind an end group that accepts it (`C[>]` caps `<`, `[<]C` caps `>`) |
| `PolymerError` of kind `InvalidMolecule` | The model itself cannot be sampled: a first object needs an end group to start from, and a terminal (`{[<] … [>]}`) must face a part |
| `PolymerError` of kind `Unsized` | A stochastic object has no `size`; give it a `ChainSize` (or a `\|…\|` distribution in the string) |
| The system total is above the amount asked for | Expected: sampling stops at the first chain reaching it, and nothing is trimmed |
| `assemble` raises "has no port for every bond" | Each monomer needs a `[<]` and a `[>]` descriptor; each cap needs the one its neighbour accepts |
| Close contacts after assembly | Expected: `GrowthPlacer` does not relax the chain; minimise before dynamics |
| Packing fails | Lower target density or increase `max_loops` |

See also: [Assembly](02_assembly.md), [Polymer Topologies](topology/index.md), [Force Field Typification](06_typifier.ipynb).